<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/rl/05_model_estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# R5 経験からモデルを推定する(notebook)

本文:[`05-model-estimation.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/rl/05-model-estimation.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-6):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

## 0. 環境:滑りやすい 4×4 のグリッド

- 状態:16マス(番号 0〜15)。左上(0)がスタート、右下(15)がゴール。ゴールに着いたら、そこにとどまり、報酬は 0
- 行動:0=上、1=右、2=下、3=左
- 遷移:確率 0.8 で選んだ方向に進み、確率 0.2 で4方向のどれかにランダムに進む。壁に向かうとその場にとどまる
- 報酬:1ステップごとに −1。ただし、マス 5 と 10 は「ぬかるみ」で、そこに入ると −10
- 割引率:γ = 0.95

本当のモデル `P_true`(形 `(16, 4, 16)`)と `R_true`(形 `(16, 4)`)を作る。`R_true[s, a]` は、`(s, a)` でもらう報酬の期待値である。

In [ ]:
N_ROW = N_COL = 4
S, A = N_ROW * N_COL, 4
GOAL, MUD = 15, {5, 10}
gamma, slip = 0.95, 0.2
MOVES = [(-1, 0), (0, 1), (1, 0), (0, -1)]          # 上・右・下・左

def move(s, a):
    r, c = divmod(s, N_COL)
    dr, dc = MOVES[a]
    r2, c2 = r + dr, c + dc
    return r2 * N_COL + c2 if (0 <= r2 < N_ROW and 0 <= c2 < N_COL) else s

def reward_on_enter(s2):
    return -10.0 if s2 in MUD else -1.0

P_true = np.zeros((S, A, S))
R_true = np.zeros((S, A))
for s in range(S):
    for a in range(A):
        if s == GOAL:
            P_true[s, a, s] = 1.0                    # ゴールではとどまり、報酬 0
            continue
        for a2 in range(A):
            p = (1 - slip) * (a2 == a) + slip / A
            P_true[s, a, move(s, a2)] += p
        R_true[s, a] = sum(P_true[s, a, s2] * reward_on_enter(s2) for s2 in range(S))

check("遷移確率の和は 1", P_true.sum(axis=2), np.ones((S, A)))

def step(s, a, rng):
    # 本当の環境で1ステップ動かす(推定には、これで集めた経験だけを使う)
    if s == GOAL:
        return GOAL, 0.0
    s2 = int(rng.choice(S, p=P_true[s, a]))
    return s2, reward_on_enter(s2)

価値反復(R4)と、方策の価値を連立方程式で求める関数(M4 §4)は用意してある。

In [ ]:
def value_iteration(P, R, gamma, tol=1e-10):
    V = np.zeros(P.shape[0])
    while True:
        Q = R + gamma * P @ V                        # 形 (S, A)
        V_new = Q.max(axis=1)
        if np.abs(V_new - V).max() < tol:
            return V_new, Q.argmax(axis=1)
        V = V_new

def policy_value(P, R, policy, gamma):
    # 方策 policy(各状態で選ぶ行動)の価値を、(I - gamma P_pi) v = r_pi で求める
    idx = np.arange(P.shape[0])
    P_pi, r_pi = P[idx, policy], R[idx, policy]
    return np.linalg.solve(np.eye(P.shape[0]) - gamma * P_pi, r_pi)

V_star, pi_star = value_iteration(P_true, R_true, gamma)
check("最適方策の価値は V* と一致", policy_value(P_true, R_true, pi_star, gamma), V_star)
arrows = np.array(["↑", "→", "↓", "←"])
print("本当のモデルでの最適方策(G=ゴール, ぬかるみ=5,10):")
grid = arrows[pi_star].astype(object); grid[GOAL] = "G"
print(grid.reshape(N_ROW, N_COL))
print("スタートの最適な価値 V*(0) =", round(V_star[0], 3))

## 1〜2. 経験を集めて、モデルを推定する

すべての `(s, a)` を順に `n_per_sa` 回ずつ試して、経験 `(s, a, r, s2)` を集める(本文 §4)。

**手計算**:本文の確認課題1(5回の経験から $\hat P$ と $\hat R$ を求める)を解く。

- 手計算:(ここに書く)

**実装**:経験のリストから、$\hat P$(形 `(S, A, S)`)と $\hat R$(形 `(S, A)`)を推定する関数 `estimate_model` を書く。一度も試していない `(s, a)` は「その場にとどまり、報酬 0」とする。

In [ ]:
def collect(n_per_sa, rng):
    data = []
    for s in range(S):
        for a in range(A):
            for _ in range(n_per_sa):
                s2, r = step(s, a, rng)
                data.append((s, a, r, s2))
    return data

def estimate_model(data, S, A):
    N_sas = np.zeros((S, A, S))
    R_sum = np.zeros((S, A))
    raise NotImplementedError("TODO: 回数 N(s,a,s2) と報酬の合計を数え、P_hat = N(s,a,s2)/N(s,a)、R_hat = 報酬の合計/N(s,a) にする。N(s,a)=0 なら「とどまる・報酬0」")
    return P_hat, R_hat

# 本文の確認課題1と同じ経験で確かめる(状態0・行動1 を5回)
toy = [(0, 1, -1.0, 1), (0, 1, -1.0, 1), (0, 1, -1.0, 1), (0, 1, -10.0, 4), (0, 1, -1.0, 4)]
P_toy, R_toy = estimate_model(toy, S, A)
check("P_hat(1|0,1) = 3/5", P_toy[0, 1, 1], 0.6)
check("P_hat(4|0,1) = 2/5", P_toy[0, 1, 4], 0.4)
check("R_hat(0,1) = -14/5", R_toy[0, 1], -14 / 5)
check("試していない (0,0) はとどまる", P_toy[0, 0, 0], 1.0)

rng = np.random.default_rng(0)
P_hat, R_hat = estimate_model(collect(200, rng), S, A)
check("推定した確率の和は 1", P_hat.sum(axis=2), np.ones((S, A)))
print("P の推定の最大誤差:", round(np.abs(P_hat - P_true).max(), 3),
      "  R の推定の最大誤差:", round(np.abs(R_hat - R_true).max(), 3))

## 3. 推定したモデルで計画し、本当のモデルで評価する

**予想**:推定したモデルで価値反復をして得た「推定モデルの中での価値」と、その方策を本当のモデルで評価した「本当の価値」は、どちらが大きくなりやすいか。

- 予想:(ここに書く)

**実装**:経験から方策を求め、その本当の価値 $v^{\hat\pi}(0)$ と、推定モデルの中での価値 $\hat V(0)$ を返す関数を書く。

In [ ]:
def plan_from_data(data):
    raise NotImplementedError("TODO: estimate_model → value_iteration で方策 pi_hat と推定モデル内の価値 V_hat を求め、pi_hat の本当の価値を policy_value で計算する")
    return pi_hat, V_hat[0], v_true[0]

for n in [1, 5, 50]:
    pi_hat, v_model, v_real = plan_from_data(collect(n, np.random.default_rng(0)))
    print(f"各(s,a)を {n:3d} 回: 推定モデル内の価値 {v_model:8.3f}   本当の価値 {v_real:8.3f}   最適 {V_star[0]:.3f}")

推定モデルの中での価値は、本当の価値と一致しない。しかも、本当の価値より**高く**出ることが多い。価値反復は、推定したモデルの中で「いちばん良く見える」行動を選ぶので、たまたま良く推定された部分(たとえば、ぬかるみに入る回数が少なく数えられた行動)を選びやすいからである。だから、方策の良さは本当の環境(ここでは本当のモデル)で評価する。

## 4. 経験の量と方策の良さ(seed を変えて)

各 `(s, a)` を試す回数を変え、seed 0〜9 の10回で「本当の価値の損失」$V^\ast(0) - v^{\hat\pi}(0)$ の平均とばらつきを調べる。経験の総数は `16 × 4 × n_per_sa` である。

In [ ]:
ns = [1, 2, 5, 10, 20, 50, 100]
loss_mean, loss_std = [], []
for n in ns:
    losses = [V_star[0] - plan_from_data(collect(n, np.random.default_rng(seed)))[2] for seed in range(10)]
    loss_mean.append(np.mean(losses)); loss_std.append(np.std(losses))
    print(f"各(s,a)を {n:3d} 回(経験 {S * A * n:5d} 個): 損失 平均 {np.mean(losses):.3f}  標準偏差 {np.std(losses):.3f}")

plt.figure(figsize=(5.5, 3.5))
plt.errorbar([S * A * n for n in ns], loss_mean, yerr=loss_std, marker="o", capsize=3)
plt.xscale("log"); plt.xlabel("number of experiences"); plt.ylabel("V*(0) - v^pi_hat(0)"); plt.grid(alpha=0.3); plt.show()

経験が増えると、損失はばらつきながら 0(最適)に近づく。経験が少ないと、seed によって結果が大きくばらつき、平均も回数の順に単調には下がらないことがある。1つの seed や1つの条件だけで判断してはいけない理由が、ここにも現れている。

## 5. 条件を変える

1. `slip` を 0.0 にすると(決定的な環境)、各 `(s, a)` を1回試すだけで最適方策が得られることを確かめる。なぜか
2. `estimate_model` で、試していない `(s, a)` を「報酬 +100、その場にとどまる」(楽観的)にし、経験を少なく(例:状態0〜7 だけを試す)したとき、方策がどう変わるかを見る
3. ぬかるみの報酬を −10 から −3 に変えると、最適方策はどう変わるか。推定にはどのくらいの経験が要るか

## 6. 説明(自分の言葉で書く)

1. 推定したモデルで計算した価値を、そのまま「方策の良さ」として報告してはいけないのはなぜか
   - (ここに書く)
2. 経験の量が同じでも、seed によって方策の良さが違うのはなぜか
   - (ここに書く)
3. 試していない `(s, a)` をどう扱うかで、エージェントの振る舞いはどう変わるか
   - (ここに書く)